# VLM-DENTAL — Zero-Shot Commercial & Base VLM Baseline Evaluation (§6, Baseline #1)

This notebook orchestrates zero-shot diagnostic benchmark evaluation on dental panoramic radiographs
without domain fine-tuning and without tool access. It evaluates general-purpose Vision-Language Models
(**Gemini**, **NVIDIA NIM**, **Groq**, **OpenRouter**, **OpenAI**, **Anthropic**, or local **vLLM** base checkpoints).

### Architecture Overview:
- **Baseline #1 in Research Protocol**: Evaluates raw VLM capability on panoramic dental X-rays to establish a non-agentic benchmark.
- **Clinical Evaluation Metrics**:
  - **FDI Tooth Localization Accuracy**: Matches quadrant (1-4) and tooth position (1-8) strictly using `dentex_row_to_fdi`.
  - **Pathology Diagnosis Accuracy & Macro-F1**: Caries, Deep Caries, Periapical Lesion, Impacted Tooth.
  - **Exact Match Accuracy with 95% Bootstrap Confidence Intervals**.
  - **Expected Calibration Error (ECE)**: Assessing model confidence calibration.
  - **Comparison against Majority-Class Baseline floor** (§20).
- **Multi-Provider Support**: Switch seamlessly between Gemini, NVIDIA NIM, Groq, OpenRouter, OpenAI, Anthropic, or local vLLM.
- **Single Run or Horizontal Slicing**: Run across all images at once (`TOTAL_SLICES = 1`) or distribute across parallel Colab/Kaggle workers (`--total-slices`, `--slice-index`, `--slice-seed`) with safe union-merging via `dental_agent/training/git_sync.py`.
- **Standard Naming Convention**: Results are automatically saved to `data/evaluations/zero_shot_{dataset}_{split}_{provider}_{model}.jsonl` for consistent downstream analysis.
- **Fail-Fast Provider Skip**: If any provider hits a 429 rate limit or quota ceiling, it immediately advances to the next provider, skipping remaining models on that exhausted provider.
- **Automatic Background Sync**: Syncs all evaluation results, LaTeX publication tables, and Markdown summaries directly to Git at the end of runs (ideal for Kaggle/Colab background execution).
- **Fully Resumable**: Skips already processed images automatically.

## 1. Mount Google Drive & Setup Workspace
Mounts Drive to access persistent storage and clones/pulls the latest `VLM-DENTAL` repository.

In [ ]:
import os, sys

# ============================================================
#  AUTO-DETECTION: Kaggle vs. Google Colab vs. Local PC
# ============================================================
IS_KAGGLE = True
IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    work_dir = '/content/VLM-DENTAL'
    if not os.path.exists(work_dir):
        os.chdir('/content')
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
    os.chdir(work_dir)
    os.system('git pull')
elif IS_KAGGLE:
    # Kaggle environment
    kaggle_dir = '/kaggle/working/VLM-DENTAL'
    if not os.path.exists(kaggle_dir) and not os.path.exists('.git'):
        os.chdir('/kaggle/working')
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
    if os.path.exists(kaggle_dir):
        os.chdir(kaggle_dir)
    os.system('git pull')
    work_dir = os.getcwd()
else:
    # Local PC environment: ensure we are in the repo
    if not os.path.exists('VLM-DENTAL') and not os.path.exists('.git'):
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
        os.chdir('VLM-DENTAL')
    elif os.path.exists('VLM-DENTAL'):
        os.chdir('VLM-DENTAL')
    
    os.system('git pull')
    work_dir = os.getcwd()

print(f'Active working directory: {os.getcwd()}')
platform_name = "Kaggle" if IS_KAGGLE else ("Google Colab" if IS_COLAB else "Local PC")
print(f'Platform: {platform_name}')


## 2. Fast Dependency Check & Installation
Installs repository dependencies, `vLLM` (for local open-source inference), API clients (`google-genai`, `openai`, `anthropic`), `tabulate`, and `scikit-learn` for metrics.

> **Note on Runtime Restart:** If Colab prompts you with *"Restart runtime to use newly installed packages"*, click **Restart Session**, then continue directly from **Cell 3**.

In [ ]:
import sys
import subprocess
import platform
import importlib.util
import importlib.metadata
import re

is_windows = platform.system() == "Windows"

# Check if transformers version is at least 5.0.0 (required for Qwen3.5 architecture)
tf_needs_upgrade = False
installed_tf_ver = "not installed"
try:
    installed_tf_ver = importlib.metadata.version("transformers")
    ver_nums = tuple(map(int, re.findall(r'\d+', installed_tf_ver)[:2]))
    if ver_nums < (5, 0):
        tf_needs_upgrade = True
except Exception:
    tf_needs_upgrade = True

# Required dependencies: (pip_package_name, python_module_name)
REQUIRED_PACKAGES = [
    ("transformers>=5.0.0", "transformers"),
    ("accelerate>=0.26.0", "accelerate"),
    ("qwen-vl-utils>=0.0.8", "qwen_vl_utils"),
    ("torchvision", "torchvision"),
    ("bitsandbytes", "bitsandbytes"),
    ("openai>=1.0.0", "openai"),
    ("google-genai>=0.1.0", "google.genai"),
    ("anthropic>=0.20.0", "anthropic"),
    ("tabulate", "tabulate"),
    ("scikit-learn", "sklearn"),
    ("python-dotenv", "dotenv"),
]

missing_packages = [
    pkg for pkg, mod in REQUIRED_PACKAGES
    if importlib.util.find_spec(mod) is None
]

if tf_needs_upgrade and "transformers>=5.0.0" not in missing_packages:
    missing_packages.insert(0, "transformers>=5.0.0")

if not missing_packages and not tf_needs_upgrade:
    print(f"✅ All required dependencies (Transformers {installed_tf_ver}, Accelerate, & API clients) are already installed. (Fast boot!)")
else:
    print(f"⚠️ Package installations/upgrades required: {missing_packages}")
    print(f"   (Detected transformers version: {installed_tf_ver}, requires >=5.0.0 for Qwen3.5)")
    print("🚀 Installing/upgrading packages...")
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "--quiet"] + missing_packages
    res = subprocess.run(cmd)
    if res.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade"] + missing_packages)
        
    # Editable install of dental_agent repository
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."])
    print("✅ Dependency installation complete!")
    if tf_needs_upgrade and "transformers" in sys.modules:
        print("\n⚠️ NOTE: Transformers was upgraded while loaded in memory.")
        print("   If you encounter any module reload warnings, please click 'Restart Session' at the top of the notebook.")


## 3. Configure Credentials (.env as Single Source of Truth)
Automatically loads credentials in two layers:
1. **`.env` file (Primary)** — Auto-searches `/content/drive/MyDrive/VLM-DENTAL/.env` or the project root. If found, this is the **single source of truth** and Colab/Kaggle Secrets are ignored.
2. **Colab Secrets Tab / Kaggle Secrets (Fallback)** — If no `.env` is found, secrets from environment variables or `google.colab.userdata` are injected.

In [ ]:
import os, shutil
from dotenv import load_dotenv

# Layer 1: .env File
candidate_env_paths = [
    os.path.join(os.getcwd(), '.env'),
    '/content/drive/MyDrive/VLM-DENTAL/.env',
    '/content/drive/MyDrive/vlmdental/.env',
    '/content/drive/MyDrive/.env',
    '/content/VLM-DENTAL/.env',
    '/content/.env',
    '/kaggle/working/VLM-DENTAL/.env',
    '/kaggle/working/.env',
]

found_env = next((p for p in candidate_env_paths if os.path.exists(p) and os.path.getsize(p) > 0), None)
if found_env:
    local_env = os.path.join(os.getcwd(), '.env')
    if os.path.abspath(found_env) != os.path.abspath(local_env):
        shutil.copy(found_env, local_env)
        print(f'Copied .env from {found_env} to {local_env}')
    load_dotenv(local_env, override=True)
    print(f'Loaded credentials from single source of truth: {found_env}')
    print('Skipping Secrets injection since .env exists.')
else:
    if not os.path.exists('.env') and os.path.exists('.env.example'):
        shutil.copy('.env.example', '.env')
        print('Created empty .env from .env.example template')
    print('No populated .env found — falling back to Colab / Kaggle Secrets...')

    # Layer 2: Overlay Colab / Kaggle Secrets (ONLY if .env was not found)
    secret_keys = [
        'GEMINI_API_KEY', 'NVIDIA_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY',
        'OPENAI_API_KEY', 'ANTHROPIC_API_KEY',
        'ZERO_SHOT_PROVIDER', 'ZERO_SHOT_MODEL',
        'NVIDIA_COOLDOWN_SECONDS', 'NVIDIA_RPD_LIMIT',
        'GROQ_COOLDOWN_SECONDS', 'GROQ_RPD_LIMIT',
        'OPENROUTER_COOLDOWN_SECONDS', 'OPENROUTER_RPD_LIMIT',
        'GEMINI_COOLDOWN_SECONDS', 'GEMINI_RPD_LIMIT',
        'HF_TOKEN', 'GITHUB_TOKEN', 'DENTEX_IMAGES_REPO'
    ]
    try:
        from google.colab import userdata
        overridden = []
        for key in secret_keys:
            try:
                val = userdata.get(key)
                if val:
                    os.environ[key] = val
                    overridden.append(key)
            except Exception:
                pass
        if overridden:
            print(f'Colab Secrets injected {len(overridden)} variable(s): {overridden}')
    except ImportError:
        pass

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        overridden = []
        for key in secret_keys:
            try:
                val = user_secrets.get_secret(key)
                if val:
                    os.environ[key] = val
                    overridden.append(key)
            except Exception:
                pass
        if overridden:
            print(f'Kaggle Secrets injected {len(overridden)} variable(s): {overridden}')
    except Exception:
        pass

# Print status overview
print('\n' + '=' * 50)
print('ACTIVE CONFIGURATION OVERVIEW')
print('=' * 50)
has_github = bool(os.environ.get('GITHUB_TOKEN', '').strip() and not os.environ.get('GITHUB_TOKEN', '').startswith('your_'))
print(f"GITHUB_TOKEN       : {'SET' if has_github else 'NOT SET'}")
active_providers = [
    p for p in ['GEMINI', 'NVIDIA', 'GROQ', 'OPENROUTER', 'OPENAI', 'ANTHROPIC']
    if os.environ.get(f'{p}_API_KEY', '').strip() and not os.environ.get(f'{p}_API_KEY', '').startswith('your_')
]
print(f"Active API Keys    : {active_providers if active_providers else 'NONE (local vLLM mode or add API keys in .env)'}")
print('=' * 50)


## 4. Model Cache Directory Setup
Configures `HF_HOME` for model weights. In Colab mode, uses the fast local SSD (`/content/`). On a local PC, uses `data/models/vllm_cache/`.

In [ ]:
import os

# ============================================================
# Platform Configuration (Auto-detected with manual override)
# ============================================================
FORCE_PLATFORM = None  # None (auto-detect), 'colab', 'kaggle', or 'local'

IS_COLAB = os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if FORCE_PLATFORM == 'colab' or (FORCE_PLATFORM is None and IS_COLAB):
    PLATFORM = 'colab'
    vllm_cache_dir = '/content/local_vllm_cache'
elif FORCE_PLATFORM == 'kaggle' or (FORCE_PLATFORM is None and IS_KAGGLE):
    PLATFORM = 'kaggle'
    vllm_cache_dir = '/kaggle/working/local_vllm_cache'
else:
    PLATFORM = 'local'
    vllm_cache_dir = os.path.join('data', 'models', 'vllm_cache')

os.makedirs(vllm_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = vllm_cache_dir
os.environ['HF_HUB_CACHE'] = os.path.join(vllm_cache_dir, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(vllm_cache_dir, 'hub')

print(f'Active Platform: {PLATFORM.upper()} | Cache location: {vllm_cache_dir}')


## 4b. Choose Dataset & Evaluation Split
- `DATASET_NAME = "dentex"` is the primary benchmark dataset.
- `SPLIT = "test"` (or `"validation"` / `"train"`):
  - **`test` / `validation`**: The official held-out test cohort (`validation_triple.json`, **50 images**) with complete 3-level diagnosis annotations.
  - **`train`**: The entire training pool (**705 images**) if you want a large-scale zero-shot evaluation pass.

In [ ]:
# ============================================================
# Dataset & Split Configuration
# ============================================================
DATASET_NAME = "dentex"   # "dentex" or "tufts"
SPLIT = "test"            # "test" (50 images), "validation" (50 images), or "train" (705 images)

print(f"Active dataset : {DATASET_NAME}")
print(f"Active split   : {SPLIT}")


## 5. Download DENTEX Dataset (If Not Already Present)

> [!NOTE]
> **Dynamic Slice Downloading:** If `DENTEX_IMAGES_REPO` is configured in your `.env`, leave `DOWNLOAD_FULL_DATASET = False`. The evaluation script will automatically download only the specific images needed for your active slice dynamically!


In [ ]:
# ============================================================
# Set DOWNLOAD_FULL_DATASET = True only if you don't have DENTEX_IMAGES_REPO
# ============================================================
DOWNLOAD_FULL_DATASET = False

if DOWNLOAD_FULL_DATASET:
    !python scripts/download_and_cleanup.py
else:
    print("Skipping full dataset download. (Zero-Shot script will fetch slice images dynamically)")

# Ensure evaluation and experiment directories exist
os.makedirs('data/evaluations', exist_ok=True)
os.makedirs('experiments', exist_ok=True)


## 6. (Optional) Local Base Model Verification (Transformers)
Verifies GPU environment, VRAM capacity, and Transformers `device_map="auto"` readiness for evaluating base checkpoints (e.g. `Qwen/Qwen3.5-9B`) directly in-process with zero API limits.

> **Note:** Skip this cell if `PROVIDER` is set to an external API (Gemini, NVIDIA NIM, Groq, OpenRouter, OpenAI, Anthropic).

In [ ]:
import sys
import os
import subprocess
import torch
import re
import importlib.metadata

# Configure PyTorch memory management for sm_75 dual T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["NCCL_P2P_DISABLE"] = "1"

# Enable SDPA memory-efficient attention for Turing sm_75
if torch.cuda.is_available():
    if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
        torch.backends.cuda.enable_mem_efficient_sdp(True)
    major_cap = torch.cuda.get_device_capability()[0]
    if major_cap < 8 and hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(False)

# Clean any legacy vLLM background processes if previously running
subprocess.run(["pkill", "-9", "-f", "vllm"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Version check for transformers
tf_ver = importlib.metadata.version("transformers") if "transformers" in sys.modules or os.path.exists(sys.executable) else "unknown"
print("=" * 65)
print("LOCAL OPEN-SOURCE VLM ENVIRONMENT (Pure FP16, SDPA Memory-Efficient Attention)")
print("=" * 65)
print(f"• Installed Transformers : v{tf_ver}")
v_nums = tuple(map(int, re.findall(r'\d+', tf_ver)[:2])) if tf_ver != "unknown" else (0, 0)
if v_nums < (5, 0):
    print("⚠️ WARNING: Transformers < 5.0.0 detected. Auto-upgrading now...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "transformers>=5.0.0"])
    print("✅ Transformers upgraded. If model loading fails, please restart the session.")

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus >= 2:
    print(f"🚀 [MULTI-GPU DETECTED] {num_gpus} GPUs detected ({num_gpus * 16}GB Total VRAM)!")
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"   • GPU {i}: {props.name} (Compute Capability {props.major}.{props.minor}, {props.total_memory / (1024**3):.1f} GB VRAM)")
    print("\n   -> Model loaded in Pure FP16 (No 4-bit, zero quantization artifacts).")
    print("   -> SDPA Memory-Efficient Attention (Cutlass sm_75) active for full resolution without OOM.")
elif num_gpus == 1:
    props = torch.cuda.get_device_properties(0)
    print(f"⚡ [SINGLE-GPU DETECTED] 1 GPU: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)")
else:
    print("⚠️ [CPU ONLY] No CUDA GPUs detected. Inference will run on CPU.")
print("=" * 65)
print("✅ Local environment ready! Proceed to Option 5 in Section 7.")


## 7. Model & Provider Configuration (Choose ONE)
Run exactly **one** of the following cells to select the VLM model(s) you want to benchmark.

- **Single Model Benchmarks**: Options 1 to 6.
- **Full Multi-Model Benchmark Suite**: Choose **Option 7** to benchmark Gemini, Groq, NVIDIA NIM, and OpenRouter in one pass!

In [ ]:
# --- Option 1: Gemini Baseline (Recommended) ---
import os

# Provider-Specific Parameters (Gemini)
os.environ["GEMINI_IMAGE_MAX_DIM"] = "0"        # 0 = unscaled full resolution
os.environ["GEMINI_MAX_TOKENS"] = "16384"       # Full token headroom

TARGET_MODELS_ARG = None
TOTAL_SLICES = 1       # Keep 1 to evaluate all images at once
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 1.5     # Seconds between API calls
GIT_SYNC_EVERY = 5     # Checkpoint git push interval (0 = disabled)
IMAGE_MAX_DIM = 0      # 0 = unscaled full resolution
MAX_TOKENS = 16384
TEMPERATURE = 0.0      # Deterministic evaluation
MAX_IMAGES = None      # None = all eligible images in split, or int (e.g. 20)

PROVIDER = "gemini"
MODEL = "gemini-3.5-flash-lite"  # Options: "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"


In [ ]:
# --- Option 2: NVIDIA NIM Baseline ---
import os

# Provider-Specific Parameters (NVIDIA NIM)
os.environ["NVIDIA_IMAGE_MAX_DIM"] = "0"        # 0 = unscaled full resolution
os.environ["NVIDIA_MAX_TOKENS"] = "16384"        # Full token headroom

TARGET_MODELS_ARG = None
TOTAL_SLICES = 1
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 1.5
GIT_SYNC_EVERY = 5
IMAGE_MAX_DIM = 0
MAX_TOKENS = 16384
TEMPERATURE = 0.0
MAX_IMAGES = None

PROVIDER = "nvidia_nim"
MODEL = "meta/llama-3.2-11b-vision-instruct"  # or "microsoft/phi-3-vision-128k-instruct"


In [ ]:
# --- Option 3: Groq Baseline (8,000 TPM Free Quota Discipline) ---
import os

# Provider-Specific Parameters (Groq: 640px image + 65s pacing strictly respects 8,000 TPM)
os.environ["GROQ_IMAGE_MAX_DIM"] = "640"   # Slashes vision tokens from ~2,200 to ~650
os.environ["GROQ_MAX_TOKENS"] = "4096"     # Full unconstrained CoT reasoning budget

TARGET_MODELS_ARG = None
TOTAL_SLICES = 1
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 65.0  # 65s sliding-window pacing refreshes Groq's 8,000 TPM rolling limit
GIT_SYNC_EVERY = 1   # Checkpoint every completed image immediately to GitHub
IMAGE_MAX_DIM = 640
MAX_TOKENS = 4096
TEMPERATURE = 0.0
MAX_IMAGES = None

PROVIDER = "groq"
MODEL = "qwen/qwen3.6-27b"


In [ ]:
# --- Option 4: OpenRouter Baseline ---
import os

# Provider-Specific Parameters (OpenRouter)
os.environ["OPENROUTER_IMAGE_MAX_DIM"] = "0"    # 0 = unscaled full resolution
os.environ["OPENROUTER_MAX_TOKENS"] = "16384"

TARGET_MODELS_ARG = None
TOTAL_SLICES = 1
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 1.5
GIT_SYNC_EVERY = 5
IMAGE_MAX_DIM = 0
MAX_TOKENS = 16384
TEMPERATURE = 0.0
MAX_IMAGES = None

PROVIDER = "openrouter"
MODEL = "google/gemma-4-31b-it:free"  # or "google/gemma-4-26b-a4b-it:free"


In [ ]:
# --- Option 5: Local Backbone (Pure FP16, Full Resolution, 16k Headroom) ---
import os, torch

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
REPETITION_PENALTY = 1.10
LOCAL_MAX_TOKENS = 16384

# Provider-Specific Parameters (Native Transformers)
os.environ["TRANSFORMERS_IMAGE_MAX_DIM"] = "0"        # 0 = unscaled full resolution (no downscaling)
os.environ["TRANSFORMERS_MAX_TOKENS"] = str(LOCAL_MAX_TOKENS)
os.environ["TRANSFORMERS_REPETITION_PENALTY"] = str(REPETITION_PENALTY)
os.environ["TRANSFORMERS_LOAD_IN_4BIT"] = "false"     # Pure FP16: strictly no 4-bit quantization

TARGET_MODELS_ARG = None
TOTAL_SLICES = 1
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 0.0     # In-process GPU inference requires 0 pacing delay
GIT_SYNC_EVERY = 5
IMAGE_MAX_DIM = 0      # 0 = unscaled full resolution
MAX_TOKENS = LOCAL_MAX_TOKENS
TEMPERATURE = 0.0
MAX_IMAGES = None

PROVIDER = "transformers"
MODEL = "Qwen/Qwen3.5-9B"               # Choice 1
# MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"  # Choice 2

print(f"Option 5 Configured:")
print(f"  • Provider       : {PROVIDER}")
print(f"  • Model          : {MODEL}")
print(f"  • Precision      : Pure FP16 (4bit=False, zero quantization)")
print(f"  • Image Dim      : Full Unscaled Resolution (IMAGE_MAX_DIM={IMAGE_MAX_DIM})")
print(f"  • Rep Penalty    : {REPETITION_PENALTY}")
print(f"  • Token Headroom : {MAX_TOKENS} tokens")


In [ ]:
# --- Option 6: OpenAI / Anthropic Commercial Baseline ---
import os

# Provider-Specific Parameters (OpenAI / Anthropic)
os.environ["OPENAI_IMAGE_MAX_DIM"] = "0"        # 0 = unscaled full resolution
os.environ["OPENAI_MAX_TOKENS"] = "4096"

TARGET_MODELS_ARG = None
TOTAL_SLICES = 1
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 1.5
GIT_SYNC_EVERY = 5
IMAGE_MAX_DIM = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.0
MAX_IMAGES = None

PROVIDER = "openai"    # or "anthropic"
MODEL = "gpt-4o"       # or "claude-3-7-sonnet-20250219"


In [ ]:
# --- Option 7: Multi-Model Benchmark Suite (Full Vision Suite) ---
import os

# Provider-Specific Constraints (Dynamically applied per provider)
# Full unscaled resolution for Gemini, NVIDIA NIM, and OpenRouter:
os.environ["GEMINI_IMAGE_MAX_DIM"] = "0"        # Gemini: full unscaled resolution
os.environ["GEMINI_MAX_TOKENS"] = "16384"
os.environ["NVIDIA_IMAGE_MAX_DIM"] = "0"        # NVIDIA NIM: full unscaled resolution
os.environ["NVIDIA_MAX_TOKENS"] = "16384"
os.environ["OPENROUTER_IMAGE_MAX_DIM"] = "0"    # OpenRouter: full unscaled resolution
os.environ["OPENROUTER_MAX_TOKENS"] = "16384"

# Groq-only TPM constraints (scaled down to fit 8000 TPM limit):
os.environ["GROQ_IMAGE_MAX_DIM"] = "640"        # Groq only: scaled down to fit 8000 TPM limit
os.environ["GROQ_MAX_TOKENS"] = "4096"          # Full reasoning budget with 65s pacing

BENCHMARK_MODELS = [
    # --- 1. Google Gemini (Lite models -- 400 RPD budget safe) ---
    ("gemini", "gemini-3.5-flash-lite"),
    ("gemini", "gemini-3.1-flash-lite"),

    # --- 2. Groq ---
    ("groq", "qwen/qwen3.6-27b"),

    # --- 3. NVIDIA NIM Vision Models ---
    ("nvidia_nim", "meta/llama-3.2-11b-vision-instruct"),
    ("nvidia_nim", "meta/llama-3.2-90b-vision-instruct"),
    ("nvidia_nim", "meta/muse-glimmer-30b"),
    ("nvidia_nim", "moonshotai/kimi-k3"),

    # --- 4. OpenRouter Free Vision Models ---
    ("openrouter", "minimax/minimax-m3:free"),
    ("openrouter", "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"),
    ("openrouter", "google/gemma-4-31b-it:free"),
    ("openrouter", "google/gemma-4-26b-a4b-it:free"),
]

TOTAL_SLICES = 1       # Keep 1 to evaluate the entire split at once
SLICE_INDEX = 1
SLICE_SEED = 42
PACING_DELAY = 1.5     # Fast pacing for Gemini/NIM/OpenRouter (Groq cooldown is handled independently)
GIT_SYNC_EVERY = 5     # Git checkpoint interval
IMAGE_MAX_DIM = 0      # 0 = unscaled full resolution (Groq uses GROQ_IMAGE_MAX_DIM dynamically)
MAX_TOKENS = None      # Dynamic per provider
TEMPERATURE = 0.0      # Deterministic evaluation
MAX_IMAGES = None      # None = all images in split (e.g. 50 in test split)

# Format into comma-separated provider:model string
TARGET_MODELS_ARG = ",".join(f"{p}:{m}" for p, m in BENCHMARK_MODELS)
PROVIDER = None
MODEL = None

print(f"Configured {len(BENCHMARK_MODELS)} benchmark targets across {len(set(p for p, _ in BENCHMARK_MODELS))} providers:")
for p, m in BENCHMARK_MODELS:
    print(f"  • [{p.upper()}] {m}")


## 8. Run Zero-Shot Baseline Evaluation
Executes `scripts/run_zero_shot.py` on the target split.
- Uses `ZERO_SHOT_PROMPT` without tool access.
- Calculates exact match, quadrant accuracy, tooth position accuracy, and pathology diagnosis accuracy.
- Appends outputs incrementally to `data/evaluations/`.
- **Fail-Fast Protection**: If a provider hits a 429/quota ceiling, it hard-skips to the next provider automatically.

In [ ]:
max_img_flag = f"--max-images {MAX_IMAGES}" if MAX_IMAGES is not None else ""
max_tok_flag = f"--max-tokens {MAX_TOKENS}" if locals().get("MAX_TOKENS") is not None else ""
rep_penalty_flag = f"--repetition-penalty {REPETITION_PENALTY}" if locals().get("REPETITION_PENALTY") is not None else ""

if locals().get('TARGET_MODELS_ARG') and TARGET_MODELS_ARG is not None:
    model_flags = f'--model "{TARGET_MODELS_ARG}"'
else:
    model_flags = f'--provider {PROVIDER} --model {MODEL}'

!python scripts/run_zero_shot.py \
    --dataset {DATASET_NAME} \
    --split {SPLIT} \
    {model_flags} \
    --total-slices {TOTAL_SLICES} \
    --slice-index {SLICE_INDEX} \
    --slice-seed {SLICE_SEED} \
    --pacing-delay {PACING_DELAY} \
    --git-sync-every {GIT_SYNC_EVERY} \
    --image-max-dim {IMAGE_MAX_DIM} \
    --temperature {TEMPERATURE} \
    {rep_penalty_flag} \
    --run-majority-baseline \
    --ignore-api-errors \
    --ignore-429 \
    {max_tok_flag} \
    {max_img_flag}


## 9. Benchmark Dashboard & Publication Tables (LaTeX / Markdown) + Auto Git Sync
Loads all zero-shot evaluation runs and formats them into a clean, comparative publication table.
Automatically saves `experiments/zero_shot_benchmark_summary.md` and `experiments/zero_shot_benchmark_summary.tex`, then commits & pushes all results to GitHub.

In [ ]:
import glob, json, sys, os
from pathlib import Path
import pandas as pd
from dental_agent.evaluation.reporting import generate_summary_table
from dental_agent.evaluation.metrics import expected_calibration_error, bootstrap_metric_ci
from dental_agent.training.git_sync import sync_and_push

eval_files = sorted(glob.glob('data/evaluations/zero_shot_*.jsonl'))
all_metrics = {}

for fpath in eval_files:
    tag = Path(fpath).stem.replace('zero_shot_', '')
    records = []
    with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    if not records:
        continue
    
    n = len(records)
    fmt_ok = sum(1 for r in records if r.get('format_ok'))
    fdi_ok = sum(1 for r in records if r.get('fdi_correct'))
    exact_ok = sum(1 for r in records if r.get('exact_match'))
    diag_ok = sum(1 for r in records if r.get('diagnosis_correct'))
    
    confidences = [r['confidence'] for r in records if r.get('confidence') is not None]
    correctness = [int(r.get('exact_match', False)) for r in records if r.get('confidence') is not None]
    ece = expected_calibration_error(confidences, correctness) if len(confidences) >= 5 else 0.0
    
    _, em_low, em_high = bootstrap_metric_ci(
        records,
        lambda recs: sum(1 for r in recs if r.get('exact_match')) / len(recs) if recs else 0.0,
    )
    
    all_metrics[tag] = {
        'format_adherence': fmt_ok / n,
        'fdi_localization_accuracy': fdi_ok / n,
        'pathology_macro_f1': diag_ok / n,
        'exact_match_accuracy': exact_ok / n,
        'exact_match_ci_95': [em_low, em_high],
        'ece': ece,
        'mean_tool_calls': 0.0,
        'total_samples': float(n),
    }

if all_metrics:
    md_table = generate_summary_table(all_metrics, table_format='github')
    latex_table = generate_summary_table(all_metrics, table_format='latex')
    
    print("\n" + "=" * 70)
    print("COMPARATIVE ZERO-SHOT BENCHMARK SUMMARY")
    print("=" * 70)
    print(md_table)
    
    print("\n" + "=" * 70)
    print("LATEX TABLE (FOR PUBLICATION)")
    print("=" * 70)
    print(latex_table)
    
    # Save summaries to experiments/
    os.makedirs('experiments', exist_ok=True)
    summary_md_path = 'experiments/zero_shot_benchmark_summary.md'
    summary_tex_path = 'experiments/zero_shot_benchmark_summary.tex'
    metrics_json_path = 'experiments/zero_shot_metrics.json'
    
    with open(summary_md_path, 'w', encoding='utf-8') as f:
        f.write('# Zero-Shot VLM Benchmark Summary (§6, Baseline #1)\n\n' + md_table + '\n')
    with open(summary_tex_path, 'w', encoding='utf-8') as f:
        f.write(latex_table + '\n')
    with open(metrics_json_path, 'w', encoding='utf-8') as f:
        json.dump(all_metrics, f, indent=2)
    
    print(f"\nSaved summary tables to {summary_md_path} and {summary_tex_path}")
    
    # Automatic Git Sync for background jobs (Kaggle / Colab)
    all_sync_paths = eval_files + [summary_md_path, summary_tex_path, metrics_json_path] + glob.glob('experiments/zero_shot_report_*.md')
    if bool(os.environ.get('GITHUB_TOKEN', '').strip() and not os.environ.get('GITHUB_TOKEN', '').startswith('your_')):
        print("\n[git-sync] Pushing all benchmark evaluations, summaries, and LaTeX tables to GitHub...")
        ok = sync_and_push(all_sync_paths, commit_message='eval: automated zero-shot benchmark dashboard and publication tables sync')
        print('✅ Sync succeeded!' if ok else 'Sync complete (or nothing new to push).')
    else:
        print("\n[git-sync] GITHUB_TOKEN not set or template. Results saved locally.")
else:
    print("No evaluation files found yet. Run Step 8 first!")


## 10. Manual Git Sync & Download Helper
Uses `dental_agent/training/git_sync.py` to safely commit and push all evaluation JSONLs and experiment summaries to Git, or download them directly to your browser.

In [ ]:
import glob
import sys
sys.path.insert(0, '.')
from dental_agent.training.git_sync import sync_and_push

eval_files = glob.glob('data/evaluations/zero_shot_*.jsonl')
exp_files = glob.glob('experiments/zero_shot_*.*')
all_files_to_sync = eval_files + exp_files

if all_files_to_sync:
    ok = sync_and_push(all_files_to_sync, commit_message='eval: manual zero-shot baseline results and reports sync')
    print('Sync succeeded.' if ok else 'Sync complete (or nothing new to push).')
else:
    print('No evaluation files found to push.')


In [ ]:
# Download evaluation JSONL files directly to your browser
try:
    from google.colab import files
    import glob
    for p in glob.glob('data/evaluations/zero_shot_*.jsonl') + glob.glob('experiments/zero_shot_*.*'):
        print(f'Downloading {p}...')
        files.download(p)
except ImportError:
    print('Not in Google Colab — files are saved locally in data/evaluations/ and experiments/')
